In [2]:
import cv2
from pathlib import Path
import os

In [3]:
def video_info(path: str) -> dict:
    cap = cv2.VideoCapture(path)
    info = {
        "frame_count": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        "fps": int(cap.get(cv2.CAP_PROP_FPS)),
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    }
    cap.release()
    return info


input_info = video_info('data/input.mp4')
output_info = video_info('data/output.mp4')
print(input_info)
print(output_info)

n_frames = input_info['frame_count']
fps = input_info['fps']
width = input_info['width']
height = input_info['height']


{'frame_count': 301, 'fps': 30, 'width': 1920, 'height': 1080}
{'frame_count': 301, 'fps': 30, 'width': 1920, 'height': 1080}


## Разбиение видео на кадры

In [4]:
def decode_video(video_path: str, out_dir: str) -> list:
    # Создаем директорию для кадров
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    paths = []

    for i in range(n_frames):
        ok, frame = cap.read()
        path = os.path.join(out_dir, f"frame_{i:03d}.jpg")
        cv2.imwrite(path, frame)
        paths.append(path)

    cap.release()
    return paths


# Разбиваем оба видео на кадры
input_frame_paths = decode_video('data/input.mp4', './frames/input')
output_frame_paths = decode_video('data/output.mp4', './frames/output')

print("Извлечено кадров из input:", len(input_frame_paths))
print("Извлечено кадров из output:", len(output_frame_paths))

Извлечено кадров из input: 301
Извлечено кадров из output: 301


## Прасим xml файл

In [7]:
import xml.etree.ElementTree as ET


def get_xml_frame_range(xml_path: str) -> tuple:
    root = ET.parse(xml_path).getroot()
    frame_ids = [int(box.attrib['frame']) for box in root.findall('.//track/box')]
    return min(frame_ids), max(frame_ids)


xml_range = get_xml_frame_range('data/annotations.xml')
print(f'Диапазон кадров в XML: {xml_range[1]}')
print(f'В видео всего кадров: {n_frames}')


Диапазон кадров в XML: 900
В видео всего кадров: 301


In [8]:
def clip_box(raw_box: tuple, image_width: int, image_height: int):
    x1, y1, x2, y2 = raw_box
    x1 = max(0, min(int(round(x1)), image_width - 1))
    y1 = max(0, min(int(round(y1)), image_height - 1))
    x2 = max(0, min(int(round(x2)), image_width))
    y2 = max(0, min(int(round(y2)), image_height))

    if x2 <= x1 or y2 <= y1:
        return None
    return [x1, y1, x2, y2]


def load_cvat_tracks(xml_path: str, total_frames: int, image_width: int, image_height: int):
    root = ET.parse(xml_path).getroot()
    valid_frame_ids = set(range(total_frames))

    labels = [node.findtext('name') for node in root.findall('.//meta/task/labels/label')]
    objects_by_frame = {frame_idx: [] for frame_idx in range(total_frames)}
    skipped_boxes = 0

    for track_node in root.findall('track'):
        track_id = int(track_node.get('id', -1))
        label = track_node.get('label', 'unknown')

        for box_node in track_node.findall('box'):
            if box_node.get('outside') == '1':
                continue

            frame_idx = int(box_node.get('frame', -1))
            if frame_idx not in valid_frame_ids:
                skipped_boxes += 1
                continue

            raw_box = (float(box_node.attrib['xtl']), float(box_node.attrib['ytl']), float(box_node.attrib['xbr']),
                       float(box_node.attrib['ybr']),
                       )
            bbox = clip_box(raw_box, image_width, image_height)
            if bbox is None:
                skipped_boxes += 1
                continue
            objects_by_frame[frame_idx].append({'bbox_xyxy': bbox, 'label': label, 'track_id': track_id, })
    summary = {
        'labels': labels,
        'boxes': sum(len(items) for items in objects_by_frame.values()),
        'skipped': skipped_boxes,
        'annotated_frames': sum(bool(items) for items in objects_by_frame.values()),
    }
    return objects_by_frame, summary


frames, ano_info = load_cvat_tracks('data/annotations.xml', n_frames, width, height)

xml_boxes = ano_info['boxes'] + ano_info['skipped']
print(f'Всего bbox в XML: {xml_boxes}')
print(f'Валидных bbox: {ano_info["boxes"]}')
print(f'Отброшено: {ano_info["skipped"]}')
print(f'Размеченных кадров: {ano_info["annotated_frames"]} из {n_frames}')
print(f'Классы: {ano_info["labels"]}')

boxes_per_frame = [len(items) for items in frames.values() if items]
print(f'В среднем объектов на размеченный кадр: {sum(boxes_per_frame) / len(boxes_per_frame):.1f}')
print(f'Максимум объектов в одном кадре: {max(boxes_per_frame) if boxes_per_frame else 0}')

Всего bbox в XML: 9659
Валидных bbox: 3647
Отброшено: 6012
Размеченных кадров: 301 из 301
Классы: ['car', 'minivan']
В среднем объектов на размеченный кадр: 12.1
Максимум объектов в одном кадре: 15
